# Nour OSeMOSYS Studio

The Studio is a local web application for building, checking, solving and
analysing OSeMOSYS models. The container runs it on port 8775 and JupyterLab
proxies it:

**<http://localhost:8888/proxy/8775/>**

Open that link in a new browser tab. The Studio's cases live in
`/work/osemosys/cases`, its runs in `/work/osemosys/runs`; both are on the
persistent work volume.


In [ ]:
import urllib.request

try:
    with urllib.request.urlopen("http://127.0.0.1:8775/", timeout=5) as response:
        body = response.read(400).decode("utf-8", "replace")
    print("Studio responded:", response.status)
    print(body[:200])
except Exception as exc:  # server may be disabled with START_OSEMOSYS_STUDIO=0
    print("Studio not reachable:", exc)


## Solve the reference model from Python

`engine/solve_check.py` builds the LP with GLPK and solves it. The built-in
standard model plus the gas-and-solar example is the smallest end-to-end
check that the solver chain works.


In [ ]:
import subprocess, sys
from pathlib import Path

STUDIO = Path("/work/models/osemosys-studio")
model = STUDIO / "web/data/builtin_model.txt"
data = STUDIO / "web/data/examples/gas_solar.txt"

result = subprocess.run(
    [sys.executable, str(STUDIO / "engine/solve_check.py"), str(model), str(data)],
    capture_output=True,
    text=True,
    timeout=600,
)
print(result.stdout[-4000:])
print(result.stderr[-2000:])


## Solvers

The `osemosys` environment ships three solvers: GLPK (`swiglpk`), CBC
(`coin-or-cbc` through PuLP) and HiGHS (`highspy`). The Studio detects them
from the environment; no extra configuration is needed.


In [ ]:
import highspy, pulp, swiglpk

print("HiGHS  :", highspy.__file__)
print("PuLP   :", pulp.__version__, "CBC available:", pulp.PULP_CBC_CMD(msg=False).available())
print("GLPK   :", swiglpk.__file__)
